# LabPulse — Colab LoRA → GGUF (llama.cpp)

Fine-tune **Qwen2.5-1.5B-Instruct** with LoRA on the CS lab-report dataset, merge adapters, export **GGUF Q4_K_M**, then download `labpulse-1.5b-q4_k_m.gguf` into your laptop `models/` folder.

**Runtime:** Colab → Runtime → Change runtime type → **T4 GPU** (or better).

**After this notebook:** local app loads only that one GGUF via llama.cpp (Windows + macOS).

## 0) Config

In [ ]:
from pathlib import Path

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 3072
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EPOCHS = 2
LR = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 4
SEED = 42

WORK = Path("/content/labpulse")
DATA_DIR = WORK / "data"
ADAPTER_DIR = WORK / "adapters"
MERGED_DIR = WORK / "merged_hf"
GGUF_DIR = WORK / "gguf"
GGUF_OUT = GGUF_DIR / "labpulse-1.5b-q4_k_m.gguf"

for p in (WORK, DATA_DIR, ADAPTER_DIR, MERGED_DIR, GGUF_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("WORK =", WORK)

## 1) Install dependencies (Unsloth + training stack)

Re-run once per new Colab runtime. Ignore pip dependency resolver warnings unless the next cell fails.

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Core training stack
pip("unsloth")
pip("datasets", "trl", "peft", "accelerate", "bitsandbytes", "transformers", "sentencepiece", "protobuf")

import torch
assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime → Change runtime type → T4)"
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

## 2) Load dataset

Pick **one** option below.

### Option A — Upload `train.jsonl` / `valid.jsonl` from your Mac
Files live in the LabPulse repo under `data/`.

In [ ]:
# Option A: upload from local machine (skip if using Option B/C)
USE_UPLOAD = True  # set False if cloning git or using Drive

if USE_UPLOAD:
    from google.colab import files
    print("Upload train.jsonl and valid.jsonl")
    uploaded = files.upload()
    for name, blob in uploaded.items():
        dest = DATA_DIR / Path(name).name
        dest.write_bytes(blob)
        print("saved", dest, len(blob), "bytes")

### Option B — Clone your LabPulse git repo

In [ ]:
# Option B: set your repo URL then run
REPO_URL = ""  # e.g. "https://github.com/YOU/Labpulse.git"

if REPO_URL:
    import shutil
    repo = Path("/content/Labpulse")
    if repo.exists():
        shutil.rmtree(repo)
    !git clone --depth 1 {REPO_URL} {repo}
    for split in ("train.jsonl", "valid.jsonl", "test.jsonl"):
        src = repo / "data" / split
        if src.exists():
            (DATA_DIR / split).write_bytes(src.read_bytes())
            print("copied", split)

### Option C — Google Drive folder with `data/`

In [ ]:
# Option C: Drive path containing train.jsonl / valid.jsonl
DRIVE_DATA = ""  # e.g. "/content/drive/MyDrive/Labpulse/data"

if DRIVE_DATA:
    from google.colab import drive
    drive.mount("/content/drive")
    src = Path(DRIVE_DATA)
    for split in ("train.jsonl", "valid.jsonl", "test.jsonl"):
        p = src / split
        if p.exists():
            (DATA_DIR / split).write_bytes(p.read_bytes())
            print("copied", split)

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path: Path):
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            assert "messages" in obj, "each line must have messages[]"
            rows.append({"messages": obj["messages"]})
    return rows

train_path = DATA_DIR / "train.jsonl"
valid_path = DATA_DIR / "valid.jsonl"
assert train_path.exists(), f"Missing {train_path} — use Option A/B/C above"
assert valid_path.exists(), f"Missing {valid_path}"

train_rows = load_jsonl(train_path)
valid_rows = load_jsonl(valid_path)
print(f"train={len(train_rows)} valid={len(valid_rows)}")
print("sample roles:", [m["role"] for m in train_rows[0]["messages"]])

## 3) Load Qwen2.5-1.5B + LoRA (Unsloth)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto
    load_in_4bit=True,   # saves Colab VRAM
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

print("trainable params ready")

## 4) Format chat examples for SFT

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

train_ds = Dataset.from_list(train_rows).map(formatting_prompts_func, batched=True)
valid_ds = Dataset.from_list(valid_rows).map(formatting_prompts_func, batched=True)
print(train_ds[0]["text"][:400], "...")

## 5) Train LoRA

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=SFTConfig(
        output_dir=str(ADAPTER_DIR / "checkpoints"),
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.03,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=200,
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=SEED,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
    ),
)

train_result = trainer.train()
print(train_result.metrics)

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("saved adapters →", ADAPTER_DIR)

## 6) Quick smoke generation (JSON check)

In [ ]:
FastLanguageModel.for_inference(model)

sample_messages = train_rows[0]["messages"][:2]  # system + user only
prompt = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(text[:1500])

# Best-effort JSON parse
try:
    start, end = text.find("{"), text.rfind("}")
    parsed = json.loads(text[start:end+1])
    print("JSON keys:", list(parsed.keys()))
except Exception as e:
    print("JSON parse note:", e)

## 7) Merge LoRA → full HF weights (16-bit)

In [ ]:
from pathlib import Path
import json
import shutil

WORK = Path("/content/labpulse")
ADAPTER_DIR = WORK / "adapters"
MERGED_DIR = WORK / "merged_hf"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 3072

assert ADAPTER_DIR.exists(), f"Missing adapters at {ADAPTER_DIR} — re-run the Train LoRA cell"
print("adapters:", [p.name for p in ADAPTER_DIR.iterdir()][:10])

# Clean incomplete merge folder
if MERGED_DIR.exists():
    shutil.rmtree(MERGED_DIR)
MERGED_DIR.mkdir(parents=True, exist_ok=True)

from unsloth import FastLanguageModel
from peft import PeftModel

# Load base in 16-bit, attach adapters, merge, save full HF folder
base, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
)
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model = model.merge_and_unload()

model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))

cfg = MERGED_DIR / "config.json"
assert cfg.exists(), "Merge failed: config.json missing"
print("config model_type:", json.loads(cfg.read_text()).get("model_type"))
print("merged files:", sorted(p.name for p in MERGED_DIR.iterdir())[:20])
print("OK merged HF model →", MERGED_DIR)

## 8) Convert merged model → GGUF Q4_K_M (llama.cpp)

Uses official `llama.cpp` conversion + quantize tools.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Re-bind paths if earlier config cell state was lost
WORK = Path("/content/labpulse")
MERGED_DIR = WORK / "merged_hf"
GGUF_DIR = WORK / "gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)
GGUF_OUT = GGUF_DIR / "labpulse-1.5b-q4_k_m.gguf"
FP16_GGUF = GGUF_DIR / "labpulse-1.5b-f16.gguf"

LLAMA_CPP = Path("/content/llama.cpp")
if not LLAMA_CPP.exists():
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp {LLAMA_CPP}

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "gguf", "numpy", "sentencepiece", "protobuf",
])

cfg = MERGED_DIR / "config.json"
assert cfg.exists(), (
    f"Missing {cfg}. Re-run the MERGE cell first until it prints OK merged HF model."
)
print("Using merged model:", MERGED_DIR)
print("files:", sorted(p.name for p in MERGED_DIR.iterdir())[:20])

# HF → GGUF (f16) — must succeed before quantize
subprocess.check_call([
    sys.executable,
    str(LLAMA_CPP / "convert_hf_to_gguf.py"),
    str(MERGED_DIR),
    "--outfile", str(FP16_GGUF),
    "--outtype", "f16",
])
assert FP16_GGUF.exists(), FP16_GGUF

# Build quantize binary if needed
quantize_bin = LLAMA_CPP / "build" / "bin" / "llama-quantize"
if not quantize_bin.exists():
    subprocess.check_call([
        "cmake", "-S", str(LLAMA_CPP), "-B", str(LLAMA_CPP / "build"), "-DGGML_CUDA=OFF",
    ])
    subprocess.check_call([
        "cmake", "--build", str(LLAMA_CPP / "build"), "--config", "Release", "-j",
        "--target", "llama-quantize",
    ])
assert quantize_bin.exists(), quantize_bin

subprocess.check_call([
    str(quantize_bin), str(FP16_GGUF), str(GGUF_OUT), "Q4_K_M",
])
assert GGUF_OUT.exists(), GGUF_OUT
print("GGUF ready:", GGUF_OUT, "size_mb=", round(GGUF_OUT.stat().st_size / 1e6, 1))

## 9) Download artifacts to your laptop

Place the GGUF at:

```text
Labpulse/models/labpulse-1.5b-q4_k_m.gguf
```

Optional: also download `adapters/` for archival.

In [ ]:
from google.colab import files

assert GGUF_OUT.exists(), GGUF_OUT
print("Downloading", GGUF_OUT.name, "…")
files.download(str(GGUF_OUT))

# Optional zip of adapters
import shutil
adapter_zip = WORK / "labpulse_adapters"
shutil.make_archive(str(adapter_zip), "zip", ADAPTER_DIR)
files.download(str(adapter_zip) + ".zip")
print("Done. Move GGUF into Labpulse/models/ then continue with local inference (step 5).")

## Troubleshooting

| Issue | Fix |
|-------|-----|
| CUDA OOM | Lower `BATCH_SIZE` to 1, `MAX_SEQ_LENGTH` to 2048, keep `load_in_4bit=True` |
| `convert_hf_to_gguf.py` missing | Pull latest llama.cpp; path may be `convert_hf_to_gguf.py` at repo root |
| Quantize target not found | `ls llama.cpp/build/bin` and use the printed `llama-quantize` path |
| JSON still messy after 2 epochs | Train `EPOCHS=3` or bump to `Qwen2.5-3B-Instruct` if Colab VRAM allows |
| Download fails | Copy GGUF to Google Drive then fetch on laptop |